# 3. Centerline extraction with VMTK

From a binary vessel mask to a set of centerlines with radii. The stage runs entirely on the CPU with VTK and VMTK: it builds a surface, finds the ends of every vessel, traces the centerlines, splits them into branches and finally flattens everything into one array that the later stages read.

This uses the fixture segmentation and takes under a minute in fast mode.

In [ ]:
import numpy as np, nibabel as nib, matplotlib.pyplot as plt
from arterial_nb import *
from arterial.io.load_and_save_operations import load_json, load_vtkpolydata
from arterial.centerline_extraction.centerline_extractor import CenterlineExtractor
from arterial.centerline_extraction.preprocessing.utils import split_segmentation

segmentation = nib.load(fixture("segmentation.nii.gz"))
seg_array = np.asarray(segmentation.dataobj)
extractor = CenterlineExtractor(case("centerlines"), MODE, fixture("segmentation.nii.gz"), fast_segmentation=True)
show_slices(seg_array, segmentation.affine, window=(0, 1), title="input: binary vessel mask");

## Islands

A segmentation is rarely one connected piece. Each connected island is processed separately; tiny ones are dropped. Fast mode also removes the top ten percent of the volume, where the intracranial tree would make endpoint detection slow and fragile.

In [ ]:
islands = split_segmentation(seg_array, segmentation.affine, minimum_island_voxel_size=1000)
voxel_mm3 = float(np.prod(np.abs(np.diag(segmentation.affine)[:3])))
for i, island in enumerate(islands):
    print(f"island {i}: {island.sum() * voxel_mm3 / 1000:6.1f} cm3")

## Surface

Marching cubes turns each island into a triangle mesh, which is smoothed and decimated. VMTK works on this surface, not on the voxels.

In [ ]:
with timed("preprocessing"), quiet():
    extractor.perform_preprocessing()
surface = extractor.segmentation_model
print(f"{surface.GetNumberOfPoints()} points, {surface.GetNumberOfCells()} triangles, {len(extractor.segmentation_model_list)} island surfaces")
plot_polydata([surface], colors=["lightgray"], title="vessel surface");

## Endpoints and centerlines

For every island surface VMTK first extracts a coarse network to detect the free ends of the vessels. One end near the aortic arch is chosen as the start; the centerlines are then traced from it to every other endpoint as the path of maximal inscribed spheres. The sphere radius at each point becomes the vessel radius.

In [ ]:
with timed("centerline extraction"), quiet():
    extractor.perform_centerline_extraction()
for idx, model in enumerate(extractor.centerline_model_list):
    endpoints = load_json(os.path.join(extractor.endpoints_dir_path, f"endpoints_{idx}.json"))["markups"][0]["controlPoints"]
    print(f"island {idx}: {len(endpoints)} endpoints -> {model.GetNumberOfCells()} centerlines, {model.GetNumberOfPoints()} points")

In [ ]:
fig = plt.figure(figsize=(16, 9))
ax = fig.add_subplot(121, projection="3d")
plot_polydata([surface] + extractor.network_list, colors=["lightgray"] + ["C1"] * len(extractor.network_list), ax=ax, title="network used to find the endpoints")
ax = fig.add_subplot(122, projection="3d")
plot_polydata([surface] + extractor.centerline_model_list, colors=["lightgray"] + [f"C{i}" for i in range(len(extractor.centerline_model_list))], ax=ax, title="centerlines, one colour per island", linewidth=1.5)
for idx in range(len(extractor.centerline_model_list)):
    pts = np.array([p["position"] for p in load_json(os.path.join(extractor.endpoints_dir_path, f"endpoints_{idx}.json"))["markups"][0]["controlPoints"]])
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], color="red", s=20)

## Branches

The centerlines overlap where several paths share a trunk. The branch extractor splits them at bifurcations into non-overlapping branches and flags the points close to a bifurcation as *blanked*: radii there are unreliable because the inscribed sphere touches two vessels.

In [ ]:
with timed("branch extraction"), quiet():
    extractor.perform_branch_model_extraction()
from vtk.util.numpy_support import vtk_to_numpy
branch = extractor.branch_model
blanking = vtk_to_numpy(branch.GetCellData().GetArray("Blanking"))
print(f"{branch.GetNumberOfCells()} branches, {int(blanking.sum())} of them blanked (near a bifurcation)")
lines = polydata_lines(branch)
fig = plt.figure(figsize=(8, 9)); ax = fig.add_subplot(111, projection="3d")
for line, flag in zip(lines, blanking):
    ax.plot(line[:, 0], line[:, 1], line[:, 2], color="red" if flag else "C0", linewidth=2 if flag else 1)
ax.set_title("branch model: red = blanked near bifurcations"); ax.view_init(10, -60);

## The segments array

The last step flattens the branches into a plain numpy array: one row per segment holding its ordered coordinates and radii, in RAS millimetres. Duplicated points and closed loops are removed, and in extracranial mode the small vessels at the very top are dropped. This array is what vessel labelling and feature extraction read.

In [ ]:
with timed("postprocessing"), quiet():
    extractor.perform_centerline_postprocessing()
segments = extractor.centerline_segments_array
lengths = [np.linalg.norm(np.diff(c, axis=0), axis=1).sum() for c in segments[:, 0]]
radii = [r.mean() for r in segments[:, 1]]
print(f"{len(segments)} segments, {sum(len(c) for c in segments[:, 0])} points in total")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(lengths, bins=15); axes[0].set_xlabel("segment length (mm)")
axes[1].hist(radii, bins=15); axes[1].set_xlabel("mean radius (mm)")
plt.tight_layout()

In [ ]:
fig = plt.figure(figsize=(8, 9)); ax = fig.add_subplot(111, projection="3d")
for coords, r in zip(segments[:, 0], segments[:, 1]):
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=r, cmap="viridis", s=4, vmin=0.5, vmax=6)
ax.set_title(f"{len(segments)} segments coloured by radius (0.5 to 6 mm)"); ax.view_init(10, -60);

## Notes

- VMTK prints `can't reconstruct new profile` for some islands; that is a warning from the network extraction and the island still gets its centerlines.
- Branch extraction runs in a separate process because a VMTK crash there would otherwise take the whole pipeline down. If it fails for the first (largest) island the stage stops; smaller islands are skipped.
- Full mode keeps the intracranial vessels and takes much longer on the same case.